In [1]:
import json
from pyspark.sql import functions as F

# Read DQ rules from external config file
config_path = "Files/yelp/config/dq_rules.json"

with open(f"/lakehouse/default/{config_path}") as f:
    raw_config = json.load(f)

# Dynamically generate fail_cond based on rule_type
def build_fail_cond(rule):
    rule_type = rule["rule_type"]
    col = rule["column_name"]

    if rule_type == "null_check":
        return F.col(col).isNull()
    elif rule_type == "range_check":
        return (F.col(col) < rule["min_val"]) | (F.col(col) > rule["max_val"])
    else:
        raise ValueError(f"Unsupported rule_type: {rule_type}")
    
# Build DQ_CONFIG with fail_cond injected
DQ_CONFIG = {}
for table, rules in raw_config.items():
    DQ_CONFIG[table] = []
    for rule in rules:
        rule_with_cond = dict(rule)
        rule_with_cond["fail_cond"] = build_fail_cond(rule)
        DQ_CONFIG[table].append(rule_with_cond)

print(f"[DQ CONFIG] loaded {sum(len(v) for v in DQ_CONFIG.values())} rules across {len(DQ_CONFIG)} tables.")

StatementMeta(, 138291f1-ab4d-4c10-8a85-2196ca1d85fb, 3, Finished, Available, Finished, False)

[DQ CONFIG] loaded 13 rules across 5 tables.
